# Data Exploration
Inspect repository coverage, schema, and distributions.

# RepoMatch — Data Understanding & Exploratory Data Analysis

## Project Overview

RepoMatch is a GitHub repository recommendation system.

The objective of this notebook is to understand the dataset before
implementing recommendation algorithms.

## Objectives

1. Identify and load the dataset.
2. Inspect its structure and data types.
3. Analyze data quality and missing values.
4. Understand repository characteristics.
5. Analyze programming languages and topics.
6. Explore repository popularity through stars and forks.
7. Visualize the dataset using Plotly.
8. Generate a data dictionary.

## Technology Stack

- Python 3.11+
- Pandas
- NumPy
- Plotly
- Scikit-learn (later stages)

In [ ]:
import os
import csv
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

print("Libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the RepoMatch project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Most starred Github Repositories.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

print("Dataset found!")
print("File name:", DATA_PATH.name)
print("File extension:", DATA_PATH.suffix)
print("File size:", round(DATA_PATH.stat().st_size / 1024, 2), "KB")
print("Absolute path:", DATA_PATH.resolve())

In [ ]:
with open(DATA_PATH, "rb") as file:
    file_signature = file.read(20)

extension = DATA_PATH.suffix.lower()

format_mapping = {
    ".csv": "CSV - Comma Separated Values",
    ".xlsx": "Excel Workbook",
    ".xls": "Excel Legacy Workbook",
    ".json": "JSON",
    ".parquet": "Apache Parquet",
}

file_format = format_mapping.get(extension, "Unknown format")

print("Detected format:", file_format)
print("File signature:", file_signature)

if extension == ".csv":
    with open(DATA_PATH, "r", encoding="utf-8-sig", errors="replace") as file:
        sample = file.read(4096)

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
        print("Detected delimiter:", repr(dialect.delimiter))
    except csv.Error:
        print("Could not automatically detect delimiter.")

In [ ]:
if extension == ".csv":
    try:
        df = pd.read_csv(
            DATA_PATH,
            encoding="utf-8-sig",
            sep=None,
            engine="python"
        )
    except UnicodeDecodeError:
        df = pd.read_csv(
            DATA_PATH,
            encoding="cp1252",
            sep=None,
            engine="python"
        )

elif extension in [".xlsx", ".xls"]:
    df = pd.read_excel(DATA_PATH)

elif extension == ".json":
    df = pd.read_json(DATA_PATH)

elif extension == ".parquet":
    df = pd.read_parquet(DATA_PATH)

else:
    raise ValueError("Unsupported file format.")

print("Dataset successfully loaded!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
rows, columns = df.shape

print("Dataset dimensions")
print("-------------------")
print(f"Number of repositories (rows): {rows:,}")
print(f"Number of attributes (columns): {columns}")
print(f"Total number of cells: {rows * columns:,}")

In [ ]:
print("Dataset columns:")
print("----------------")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

In [ ]:
dtype_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Count": df.notna().sum().values,
    "Missing Count": df.isna().sum().values,
    "Unique Values": df.nunique().values
})

display(dtype_summary)

In [ ]:
def find_column(dataframe, candidates):
    """
    Find a column using possible column-name variations.
    Returns the actual column name or None.
    """
    normalized_columns = {
        str(col).strip().lower().replace(" ", "_"): col
        for col in dataframe.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "_")

        if key in normalized_columns:
            return normalized_columns[key]

    return None


# Identify important columns
REPO_COL = find_column(df, [
    "repository", "repository_name", "repo_name",
    "name", "full_name", "repository_full_name"
])

STARS_COL = find_column(df, [
    "stars", "star", "stargazers_count", "watchers"
])

FORKS_COL = find_column(df, [
    "forks", "fork", "forks_count"
])

LANGUAGE_COL = find_column(df, [
    "language", "programming_language", "primary_language"
])

TOPICS_COL = find_column(df, [
    "topics", "topic", "tags", "keywords"
])

DESCRIPTION_COL = find_column(df, [
    "description", "repository_description", "about"
])

CREATED_COL = find_column(df, [
    "created_at", "created", "creation_date", "created_date"
])

ARCHIVED_COL = find_column(df, [
    "archived", "is_archived"
])

detected_columns = {
    "Repository": REPO_COL,
    "Stars": STARS_COL,
    "Forks": FORKS_COL,
    "Language": LANGUAGE_COL,
    "Topics": TOPICS_COL,
    "Description": DESCRIPTION_COL,
    "Creation Date": CREATED_COL,
    "Archived": ARCHIVED_COL
}

for name, column in detected_columns.items():
    print(f"{name:20} -> {column}")

In [ ]:
number_of_repositories = len(df)

print(f"Total repositories: {number_of_repositories:,}")

In [ ]:
missing_analysis = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2),
    "Available Values": df.notna().sum()
})

missing_analysis = missing_analysis.sort_values(
    by="Missing Percentage",
    ascending=False
)

display(missing_analysis)

In [ ]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print(
    "Duplicate percentage:",
    round(duplicate_rows / len(df) * 100, 2), "%"
    if len(df) else "N/A"
)

# Display duplicate examples
if duplicate_rows > 0:
    display(df[df.duplicated(keep=False)].head(10))

In [ ]:
if REPO_COL:
    duplicate_names = df[REPO_COL].duplicated().sum()

    print("Duplicate repository names:", duplicate_names)

    if duplicate_names > 0:
        display(
            df[df[REPO_COL].duplicated(keep=False)]
            .sort_values(REPO_COL)
        )
else:
    print("Repository name column not detected.")

In [ ]:
if LANGUAGE_COL:
    languages = df[LANGUAGE_COL].dropna().astype(str).str.strip()

    unique_languages = sorted(
        languages[languages.ne("")].unique(),
        key=str.lower
    )

    print("Number of unique programming languages:", len(unique_languages))
    print("\nProgramming languages:")

    for language in unique_languages:
        print("-", language)

else:
    print("Programming language column not found.")

In [ ]:
def extract_topics(value):
    if pd.isna(value):
        return []

    if isinstance(value, (list, tuple, set)):
        return [str(x).strip() for x in value if str(x).strip()]

    text_value = str(value).strip()

    if not text_value:
        return []

    # Handle JSON-like lists
    if text_value.startswith("[") and text_value.endswith("]"):
        try:
            parsed = json.loads(text_value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except (json.JSONDecodeError, TypeError):
            pass

    # Handle common separators
    separator = "|" if "|" in text_value else ","

    return [
        topic.strip().strip("'\"")
        for topic in text_value.split(separator)
        if topic.strip().strip("'\"")
    ]


if TOPICS_COL:
    all_topics = df[TOPICS_COL].dropna().apply(extract_topics)

    flat_topics = [
        topic
        for topics in all_topics
        for topic in topics
    ]

    unique_topics = sorted(set(flat_topics), key=str.lower)

    print("Total topic occurrences:", len(flat_topics))
    print("Number of unique topics:", len(unique_topics))

    print("\nFirst 50 unique topics:")
    print(unique_topics[:50])

else:
    print("Topics column not found.")

In [ ]:
for column_name, column in [
    ("Stars", STARS_COL),
    ("Forks", FORKS_COL)
]:
    if column:
        df[column] = pd.to_numeric(df[column], errors="coerce")

        print(f"\n{column_name} Distribution")
        print("-" * 30)

        display(
            df[column].describe(
                percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
            ).to_frame()
        )

In [ ]:
if DESCRIPTION_COL:
    descriptions = df[DESCRIPTION_COL]

    missing_description = (
        descriptions.isna()
        | descriptions.astype(str).str.strip().isin(["", "nan", "None"])
    )

    count_without_description = missing_description.sum()

    percentage_without_description = (
        count_without_description / len(df) * 100
        if len(df) else 0
    )

    print("Repositories without descriptions:", count_without_description)
    print(
        "Percentage without descriptions:",
        round(percentage_without_description, 2), "%"
    )

else:
    print("Description column not found.")

In [ ]:
column_descriptions = {
    "repository": "Repository name or identifier.",
    "repository_name": "Name of the GitHub repository.",
    "repo_name": "Name of the GitHub repository.",
    "full_name": "Full GitHub repository identifier (owner/repository).",
    "stars": "Number of GitHub stars, indicating repository popularity.",
    "stargazers_count": "Number of GitHub users who starred the repository.",
    "forks": "Number of times the repository has been forked.",
    "forks_count": "Number of repository forks.",
    "language": "Main programming language used in the repository.",
    "description": "Short textual description of the repository.",
    "topics": "GitHub topics or tags describing the repository.",
    "created_at": "Date when the repository was created.",
    "updated_at": "Date when the repository was last updated.",
    "archived": "Indicates whether the repository is archived.",
    "license": "Software license associated with the repository.",
    "open_issues": "Number of currently open issues.",
    "watchers": "Number of users watching the repository.",
    "size": "Repository size, generally expressed in KB.",
    "html_url": "URL of the repository on GitHub."
}


def describe_column(column):
    normalized = str(column).strip().lower().replace(" ", "_")

    if normalized in column_descriptions:
        return column_descriptions[normalized]

    if "date" in normalized or normalized.endswith("_at"):
        return "Date or timestamp associated with the repository."

    if "url" in normalized or "link" in normalized:
        return "Web address associated with the repository."

    if "count" in normalized:
        return "Numerical count associated with the repository."

    if "id" == normalized or normalized.endswith("_id"):
        return "Unique identifier associated with the repository."

    return "Description to be verified based on the original dataset."


data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Description": [describe_column(col) for col in df.columns],
    "Non-Null Values": df.notna().sum().values,
    "Missing Values": df.isna().sum().values,
    "Unique Values": df.nunique().values,
    "Example": [
        df[col].dropna().iloc[0] if df[col].notna().any() else None
        for col in df.columns
    ]
})

display(data_dictionary)

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data_dictionary.to_csv(
    OUTPUT_DIR / "data_dictionary.csv",
    index=False
)

print("Data dictionary exported successfully!")

In [ ]:
if LANGUAGE_COL:
    language_counts = (
        df[LANGUAGE_COL]
        .dropna()
        .astype(str)
        .str.strip()
    )

    language_counts = language_counts[language_counts.ne("")]
    top_languages = language_counts.value_counts().head(10)

    fig = px.bar(
        x=top_languages.index,
        y=top_languages.values,
        title="Top 10 Programming Languages",
        labels={
            "x": "Programming Language",
            "y": "Number of Repositories"
        },
        text=top_languages.values
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Programming Language",
        yaxis_title="Number of Repositories",
        xaxis_tickangle=-35
    )

    fig.show()

else:
    print("Language column not found.")

In [ ]:
if REPO_COL and STARS_COL:
    top_repositories = (
        df[[REPO_COL, STARS_COL]]
        .dropna()
        .sort_values(STARS_COL, ascending=False)
        .head(10)
    )

    display(top_repositories)

    fig = px.bar(
        top_repositories.sort_values(STARS_COL),
        x=STARS_COL,
        y=REPO_COL,
        orientation="h",
        title="Top 10 GitHub Repositories by Stars",
        labels={
            REPO_COL: "Repository",
            STARS_COL: "Number of Stars"
        },
        text=STARS_COL
    )

    fig.update_layout(
        template="plotly_white",
        yaxis_title="Repository",
        xaxis_title="Stars"
    )

    fig.show()

else:
    print("Repository or stars column not found.")

In [ ]:
if STARS_COL:
    stars_data = df[STARS_COL].dropna()

    fig = px.histogram(
        x=stars_data,
        nbins=50,
        title="Distribution of GitHub Stars",
        labels={
            "x": "Number of Stars",
            "y": "Number of Repositories"
        },
        marginal="box"
    )

    fig.update_layout(
        template="plotly_white",
        bargap=0.05
    )

    fig.show()

    # Logarithmic distribution for better visibility
    fig_log = px.histogram(
        x=stars_data[stars_data > 0],
        nbins=50,
        log_x=True,
        title="Distribution of GitHub Stars (Logarithmic Scale)",
        labels={
            "x": "Number of Stars (Log Scale)",
            "y": "Number of Repositories"
        }
    )

    fig_log.update_layout(template="plotly_white")
    fig_log.show()

In [ ]:
if FORKS_COL:
    forks_data = df[FORKS_COL].dropna()

    fig = px.histogram(
        x=forks_data,
        nbins=50,
        title="Distribution of GitHub Forks",
        labels={
            "x": "Number of Forks",
            "y": "Number of Repositories"
        },
        marginal="box"
    )

    fig.update_layout(
        template="plotly_white",
        bargap=0.05
    )

    fig.show()

    fig_log = px.histogram(
        x=forks_data[forks_data > 0],
        nbins=50,
        log_x=True,
        title="Distribution of GitHub Forks (Logarithmic Scale)",
        labels={
            "x": "Number of Forks (Log Scale)",
            "y": "Number of Repositories"
        }
    )

    fig_log.update_layout(template="plotly_white")
    fig_log.show()

In [ ]:
if STARS_COL and FORKS_COL:
    scatter_columns = [STARS_COL, FORKS_COL]
    if REPO_COL:
        scatter_columns.append(REPO_COL)

    scatter_data = df[scatter_columns].dropna()

    fig = px.scatter(
        scatter_data,
        x=STARS_COL,
        y=FORKS_COL,
        title="Relationship Between Stars and Forks",
        labels={
            STARS_COL: "Stars",
            FORKS_COL: "Forks"
        },
        opacity=0.6,
        hover_data=[REPO_COL] if REPO_COL else None
    )

    fig.update_layout(
        template="plotly_white"
    )

    fig.show()

    correlation = scatter_data[STARS_COL].corr(
        scatter_data[FORKS_COL]
    )

    print("Pearson correlation:", round(correlation, 4))

else:
    print("Stars or forks column not found.")

In [ ]:
missing_values = df.isna().sum()

missing_values = missing_values[missing_values > 0].sort_values(
    ascending=False
)

if len(missing_values) > 0:

    fig = px.bar(
        x=missing_values.index,
        y=missing_values.values,
        title="Missing Values per Column",
        labels={
            "x": "Column",
            "y": "Number of Missing Values"
        },
        text=missing_values.values
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_tickangle=-45
    )

    fig.show()

else:
    print("No missing values found!")

In [ ]:
if CREATED_COL:

    creation_data = df[[CREATED_COL]].copy()

    creation_data[CREATED_COL] = pd.to_datetime(
        creation_data[CREATED_COL],
        errors="coerce",
        utc=True
    )

    creation_data = creation_data.dropna()

    creation_data["Creation Year"] = (
        creation_data[CREATED_COL].dt.year
    )

    repositories_per_year = (
        creation_data["Creation Year"]
        .value_counts()
        .sort_index()
        .reset_index()
    )

    repositories_per_year.columns = [
        "Year",
        "Number of Repositories"
    ]

    display(repositories_per_year)

    fig = px.line(
        repositories_per_year,
        x="Year",
        y="Number of Repositories",
        markers=True,
        title="GitHub Repository Creation Over Time"
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="Creation Year",
        yaxis_title="Number of Repositories"
    )

    fig.show()

else:
    print("Creation date column not found.")

In [ ]:
if ARCHIVED_COL:

    archived_data = (
        df[ARCHIVED_COL]
        .fillna(False)
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": "Archived",
            "1": "Archived",
            "yes": "Archived",
            "false": "Not Archived",
            "0": "Not Archived",
            "no": "Not Archived"
        })
    )

    archived_counts = archived_data.value_counts()

    fig = px.pie(
        values=archived_counts.values,
        names=archived_counts.index,
        title="Archived vs Non-Archived Repositories",
        hole=0.4
    )

    fig.update_layout(template="plotly_white")
    fig.show()

    display(archived_counts.to_frame("Number of Repositories"))

else:
    print("Archived column not available in this dataset.")

In [ ]:
summary = {
    "Total repositories": len(df),
    "Total columns": len(df.columns),
    "Duplicate rows": int(df.duplicated().sum()),
    "Total missing values": int(df.isna().sum().sum()),
    "Columns with missing values": int((df.isna().sum() > 0).sum()),
    "Unique programming languages": (
        int(df[LANGUAGE_COL].nunique())
        if LANGUAGE_COL else "Not available"
    ),
    "Unique topics": (
        len(unique_topics)
        if TOPICS_COL else "Not available"
    ),
    "Average stars": (
        round(df[STARS_COL].mean(), 2)
        if STARS_COL else "Not available"
    ),
    "Maximum stars": (
        df[STARS_COL].max()
        if STARS_COL else "Not available"
    ),
    "Average forks": (
        round(df[FORKS_COL].mean(), 2)
        if FORKS_COL else "Not available"
    )
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["Metric", "Value"]
)

display(summary_df)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(
    OUTPUT_DIR / "eda_summary.csv",
    index=False
)

missing_analysis.to_csv(
    OUTPUT_DIR / "missing_values_analysis.csv"
)

df.to_csv(
    OUTPUT_DIR / "repositories_inspected.csv",
    index=False
)

print("EDA outputs exported successfully!")
print("Output directory:", OUTPUT_DIR.resolve())